### Cài đặt và import các thư viện cần thiết

In [ ]:
!pip install unsloth vllm==0.7.3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.6/264.6 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.5/96.5 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.6/87.6 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 906.5/906.5 MB 1.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 102.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 124.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 114.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.9

In [ ]:
import os
import re

from vllm import SamplingParams
from unsloth import FastLanguageModel
from datasets import load_dataset, Dataset
from trl import GRPOConfig, GRPOTrainer

<ipython-input-1-52ca1b5955e7>:5: UserWarning: WARNING: Unsloth should be imported before transformers to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


Unsloth: Patching Xformers to fix some performance issues.
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 05-04 09:23:34 __init__.py:207] Automatically detected platform cuda.


###  Load mô hình lớn và setup cài đặt LoRA

In [ ]:
max_seq_length= 2048
lora_rank =64

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="meta-llama/Llama-3.2-1B-Instruct",
    max_seq_length=max_seq_length,
    load_in_4bit=False,
    fast_inference=True,
    max_lora_rank=lora_rank,
    gpu_memory_utilization=0.8,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=lora_rank,
    target_modules=[
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=lora_rank,
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

==((====))==  Unsloth 2025.4.7: Fast Llama patching. Transformers: 4.51.3. vLLM: 0.7.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.1.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.28.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM loading unsloth/Llama-3.2-1B-Instruct with actual GPU utilization = 79.24%
Unsloth: Your GPU has CUDA compute capability 7.5 with VRAM = 14.74 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 2048. Num Sequences = 224.
Unsloth: vLLM's KV Cache can use up to 9.3 GB. Also swap space = 2 GB.
WARNING 05-04 09:23:54 config.py:2448] Casting torch.bfloat16 to torch.float16.
INFO 05-04 09:24:08 config.py:549] This model supports multiple tasks: {'embed', 'score', 'generate', 'classify', 'reward'}. Def

tokenizer_config.json:   0%|          | 0.00/54.7k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

...

INFO 05-04 09:26:16 model_runner.py:1562] Graph capturing finished in 38 secs, took 0.20 GiB
INFO 05-04 09:26:16 llm_engine.py:436] init engine (profile, create kv cache, warmup model) took 52.62 seconds


tokenizer_config.json:   0%|          | 0.00/54.7k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

Unsloth 2025.4.7 patched 16 layers with 16 QKV layers, 16 O layers and 16 MLP layers.


### Tải bộ dữ liệu

In [ ]:
dataset = load_dataset("5CD-AI/Vietnamese-meta-math-MetaMathQA-40K-gg-translated", split="train")

README.md:   0%|          | 0.00/118 [00:00<?, ?B/s]

MetaMathQA-40K_vi.json:   0%|          | 0.00/69.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/40000 [00:00<?, ? examples/s]

### Cài đặt format prompt cho reasoning

In [ ]:
answer_pattern = re.compile(
    r"(đáp án là:|đáp án là :|câu trả lời là:|câu trả lời là :)\s*(.*)",
    re.IGNORECASE
)

formatted_dataset = []
for item in dataset:
    response = item["response_vi"].strip().lower()
    match = answer_pattern.search(response)
    if match:
        answer = match.group(2).strip()
        formatted_dataset.append({
            "question": item["query_vi"],
            "answer": answer
        })

reasoning_start = "<thinking>"
reasoning_end = "</thinking>"
solution_start = "<SOLUTION>"
solution_end = "</SOLUTION>"

system_prompt = \
    f"""You are given a problem.
Think about the problem and provide your thought process.
Place it between {reasoning_start} and {reasoning_end}.
Then, provide your final answer between {solution_start}{solution_end}"""

train_dataset = Dataset.from_list(formatted_dataset[:8000])
train_dataset = train_dataset.map(lambda x: {
    "prompt": [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": x["question"]},
    ],
    "answer": x["answer"],
})

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

### Định nghĩa các hàm reward cho học tăng cường

#### Reward cho output đúng format toán

In [ ]:
match_format = re.compile(
    rf"^[\s]{{0,}}"
    rf"{reasoning_start}.+?{reasoning_end}.*?"
    rf"{solution_start}(.+?){solution_end}"
    rf"[\s]{{0,}}\$",
    flags=re.MULTILINE | re.DOTALL
)

def match_format_exactly(completions, **kwargs):
    scores = []
    for completion in completions:
        score = 0
        response = completion[0]["content"]
        if match_format.search(response) is not None:
            score += 3.0
        scores.append(score)
    return scores

def match_format_approximately(completions, **kwargs):
    scores = []
    for completion in completions:
        score = 0
        response = completion[0]["content"]
        score += 0.5 if response.count(reasoning_start) == 1 else-1.0
        score += 0.5 if response.count(reasoning_end) == 1 else-1.0
        score += 0.5 if response.count(solution_start) == 1 else-1.0
        score += 0.5 if response.count(solution_end) == 1 else-1.0
        scores.append(score)
    return scores

#### Reward cho output đúng đáp án

In [ ]:
match_numbers = re.compile(
    solution_start + r".*?([\d\.\,]{1,})",
    flags=re.MULTILINE | re.DOTALL
)

def check_answer(prompts, completions, answer, **kwargs):
    responses = [completion[0]["content"] for completion in completions]
    extracted_responses = [
        guess.group(1)
        if (guess := match_format.search(r)) is not None else None
        for r in responses
    ]

    scores = []
    for guess, true_answer in zip(extracted_responses, answer):
        score = 0
        if guess is None:
            scores.append(0)
            continue
        if guess == true_answer:
            score += 3.0
        elif guess.strip() == true_answer.strip():
            score += 1.5
        else:
            score-= 1.5
        scores.append(score)
    return scores

def check_numbers(prompts, completions, answer, **kwargs):
    question = prompts[0][-1]["content"]
    responses =[completion[0]["content"] for completion in completions]

    extracted_responses= [
        guess.group(1)
        if (guess := match_numbers.search(r)) is not None else None
        for r in responses
    ]

    count = getattr(check_numbers, 'counter', 0) + 1
    check_numbers.counter = count
    if count %5 == 0:
        print('*'*20, f"Question:{question}", f"\nResponse:\n{responses[0]}",
              f"\nExtracted: {extracted_responses[0]}", f"\nGTAnswer: {answer[0]}")

    scores = []
    for guess, true_answer in zip(extracted_responses, answer):
        if guess is None:
            scores.append(0)
            continue
        try:
            true_answer =float(true_answer.strip())
            # Remove commas like in 123,456
            guess = float(guess.strip().replace(",", ""))
            scores.append(1.5 if guess == true_answer else -0.5)
        except:
            scores.append(0)
    return scores

### Huấn luyện mô hình

In [ ]:
max_len = max(train_dataset.map(
    lambda x: {"tokens": tokenizer.apply_chat_template(
        x["prompt"], add_generation_prompt=True, tokenize=True)},
    batched=True,
).map(lambda x: {"length": len(x["tokens"])})["length"])

max_prompt_length= max_len + 1

training_args = GRPOConfig(
    learning_rate=5e-6,
    weight_decay=5e-4,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    optim="adamw_torch_fused",
    logging_steps=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=64,
    num_generations=8,
    max_prompt_length=max_prompt_length,
    max_completion_length=max_seq_length-max_prompt_length,
    num_train_epochs=1,
    max_steps=-1,
    save_steps=250,
    max_grad_norm=0.1,
    report_to="wandb",
    output_dir="outputs_bz2",
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        match_format_exactly,
        match_format_approximately,
        check_answer,
        check_numbers,
    ],
    args=training_args,
    train_dataset=train_dataset,
)
trainer.train()

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Unsloth: We now expect `per_device_train_batch_size` to be a multiple of `num_generations`.
We will change the batch size of 2 to the `num_generations` of 8


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 8,000 | Num Epochs = 1 | Total steps = 125
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 64
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 64 x 1) = 512
 "-____-"     Trainable parameters = 45,088,768/1,280,903,168 (3.52% trained)


Unsloth: Will smartly offload gradients to save VRAM!
******************** Question:Roe tiết kiệm được $10 mỗi tháng từ tháng 1 đến tháng 7 và tiết kiệm x $ mỗi tháng từ tháng 8 đến tháng 11. Cô ấy nên tiết kiệm 20 đô la vào tháng 12 để có tổng số tiền tiết kiệm được là 150 đô la trong năm. Giá trị của biến x chưa biết là bao nhiêu? 
Response:
<thinking>

Để tìm giá trị của biến x, ta cần sử dụng phương pháp lập hệ phương trình.

Gọi số tiền mà Roe tiết kiệm được trong 12 tháng đầu là $10t và số tiền mà cô ấy tiết kiệm được trong 5 tháng sau là 20t.

Vì mà cô ấy tiết kiệm được 150 đô la tổng cộng trong 12 tháng, ta có:

10t + 20t = 150

Combinate hai mặt bằng ta có:

30t = 150

Nào số phần trăm của cô ấy tiết kiệm được 10 cho 12 tháng là (150/30) *100 = 500%
Nào ta có hoạch chia 150 cho 10 để tìm số tiền tiết kiệm được trong tháng 8, ta có 

150/10 = 15

Enter giá trị x = 15 
Extracted: None 
GTAnswer: 15
******************** Question:Tracy, John và Jake nhận thấy tổng trọng lượng của 

Step,Training Loss,reward,reward_std,completion_length,kl,rewards / match_format_exactly,rewards / match_format_approximately,rewards / check_answer,rewards / check_numbers
1,0.000000,-2.828125,0.881859,243.994141,0.000000,0.000000,-2.819336,0.000000,-0.008789
2,0.000000,-2.678711,0.804561,223.570312,0.000000,0.000000,-2.675781,0.000000,-0.002930


******************** Question:Joshua đóng gói x chai vào mỗi thùng. Anh ta có tổng cộng 130 chai và 10 thùng. Có bao nhiêu chai sẽ không được đặt trong một thùng? Nếu chúng ta biết câu trả lời cho câu hỏi trên là 10 thì giá trị của biến x chưa biết là bao nhiêu? 
Response:
<thinking>

Công thức cho số chai mỗi thùng là 130/x, 

Vậy số chai sẽ được đặt trong một thùng là 130/x ≥ 10 

Để giải một thùng, ta có thể thực hiện chia 130/x như sau:
130/10 = 13 >= x, 

Vậy có 13 chai được đặt trong một thùng. 
Extracted: None 
GTAnswer: 12
******************** Question:Một người nông dân có một thửa ruộng hình chữ nhật có kích thước $2m+7$ và $m-2$. Nếu trường có diện tích X đơn vị vuông, Giá trị của $m$ là 5. Giá trị của biến X chưa biết là bao nhiêu? 
Response:
<thinking>

Để giải quyết vấn đề này, ta cần tính diện tích của thửa ruộng. 

Diện tích của một thửa ruộng hình chữ nhật có kích thước $l \times b$ là $l \times b$, trong trường hợp này là $2\times (m - 2)$. 

Vậy diện tích của thửa ru

### Lưu trữ và inference

In [ ]:
model.save_lora("grpo_saved_lora")

In [ ]:
idx = 0
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": train_dataset[idx]["question"]},
]

sampling_params = SamplingParams(
    temperature = 0.8,
    top_p = 0.95,
    max_tokens = 1024,
)

text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    tokenize = False,
)
path_lora = "grpo_saved_lora"
output = model.fast_generate(
    [text],
    sampling_params = sampling_params,
    lora_request = model.load_lora(path_lora),
)[0].outputs[0].text

print(f"Problem:\n{train_dataset[idx]['question']}")
print(f"Response:\n{output}")
print("GT Answer:", train_dataset[idx]["answer"])